# Named Entity Recognition: DistilBERT fine-tuning
## 1. Load data, tokenize, align labels

In [ ]:
!pip install -q seqeval

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset("eriktks/conll2003", revision="refs/convert/parquet")
label_names = ds["train"].features["ner_tags"].feature.names

MODEL_NAME = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(batch):
    enc = tokenizer(batch["tokens"], is_split_into_words=True,
                    truncation=True, max_length=128)
    all_labels = []
    for i, tags in enumerate(batch["ner_tags"]):
        word_ids = enc.word_ids(batch_index=i)
        labels, prev = [], None
        for w in word_ids:
            if w is None:
                labels.append(-100)
            elif w != prev:
                labels.append(tags[w])
            else:
                labels.append(-100)
            prev = w
        all_labels.append(labels)
    enc["labels"] = all_labels
    return enc

tok_ds = ds.map(tokenize_and_align, batched=True)

keras_results = {
    "validation": {"LSTM": 0.7921, "BiLSTM": 0.8456, "BiLSTM+CRF": 0.8552, "BiLSTM+CRF+Char": 0.9113},
    "test":       {"LSTM": 0.7039, "BiLSTM": 0.7713, "BiLSTM+CRF": 0.7834, "BiLSTM+CRF+Char": 0.8551},
}
print(label_names)

## 2. Fine-tune DistilBERT with the HuggingFace Trainer

In [ ]:
import gc
import numpy as np
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)
from seqeval.metrics import f1_score, precision_score, recall_score

id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in enumerate(label_names)}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_names),
    id2label=id2label, label2id=label2id)

collator = DataCollatorForTokenClassification(tokenizer)

def compute_metrics(p):
    preds = p.predictions.argmax(axis=-1)
    true_tags, pred_tags = [], []
    for pr, lb in zip(preds, p.label_ids):
        keep = lb != -100
        true_tags.append([id2label[t] for t in lb[keep]])
        pred_tags.append([id2label[t] for t in pr[keep]])
    return {"precision": precision_score(true_tags, pred_tags),
            "recall": recall_score(true_tags, pred_tags),
            "f1": f1_score(true_tags, pred_tags)}

args = TrainingArguments(
    output_dir="ner-distilbert",
    seed=42,
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tok_ds["train"],
    eval_dataset=tok_ds["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)

gc.collect()
trainer.train()

### Evaluate DistilBERT on validation and test

In [ ]:
from seqeval.metrics import classification_report

def hf_predict(split):
    out = trainer.predict(tok_ds[split])
    preds = out.predictions.argmax(axis=-1)
    true_tags, pred_tags = [], []
    for pr, lb in zip(preds, out.label_ids):
        keep = lb != -100
        true_tags.append([id2label[t] for t in lb[keep]])
        pred_tags.append([id2label[t] for t in pr[keep]])
    return true_tags, pred_tags

for split in ["validation", "test"]:
    t, p = hf_predict(split)
    print(split.upper())
    print(classification_report(t, p, digits=4))
    keras_results[split]["DistilBERT"] = f1_score(t, p)

for split in ["validation", "test"]:
    print(split, {k: round(v, 4) for k, v in keras_results[split].items()})

trainer.save_model("distilbert-ner-final")
tokenizer.save_pretrained("distilbert-ner-final")

## 3. Gradio demo: color-coded entity spans

In [ ]:
!pip install -q gradio

In [9]:
import gradio as gr
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model="distilbert-ner-final",
    tokenizer="distilbert-ner-final",
    aggregation_strategy="first",
    device=0,
)

def predict(text):
    if not text.strip():
        return {"text": "", "entities": []}
    ents = ner(text)
    return {
        "text": text,
        "entities": [{"entity": e["entity_group"], "start": e["start"], "end": e["end"]}
                     for e in ents],
    }

COLORS = {"PER": "#f9a8d4", "ORG": "#93c5fd", "LOC": "#86efac", "MISC": "#fde68a"}

demo = gr.Interface(
    fn=predict,
    flagging_mode="never",
    inputs=gr.Textbox(lines=4, label="Text"),
    outputs=gr.HighlightedText(label="Entities", color_map=COLORS, show_legend=True),
    examples=[
        "Barack Obama visited Cairo and met officials from Microsoft.",
        "Real Madrid beat Manchester United in Lisbon on Tuesday.",
        "The United Nations said the European Union will send aid to Syria.",
    ],
    title="Named Entity Recognition (CoNLL-2003)",
)

demo.launch(share=True)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://04cbbca213de479144.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
tests = [
    "Mohamed Salah scored twice for Liverpool at Anfield.",
    "The Egyptian pound fell against the Euro on Monday.",
    "Apple released a new phone while Amazon opened a warehouse near the Amazon river.",
    "Jordan met Washington in Washington.",
    "OpenAI and Anthropic released new models in 2026.",
    "Ahmed Hassan works at Vodafone in Alexandria.",
    "barack obama visited cairo.",
    "BREAKING: NASA LAUNCHES NEW SATELLITE FROM FLORIDA",
]

for s in tests:
    print(s)
    ents = ner(s)
    if not ents:
        print("   (no entities found)")
    for e in ents:
        print(f"   {e['word']:20} {e['entity_group']:5} {e['score']:.2f}")
    print()

Mohamed Salah scored twice for Liverpool at Anfield.
   Mohamed Salah        PER   1.00
   Liverpool            ORG   0.99
   Anfield              LOC   0.98

The Egyptian pound fell against the Euro on Monday.
   Egyptian             MISC  1.00
   Euro                 MISC  0.99

Apple released a new phone while Amazon opened a warehouse near the Amazon river.
   Apple                ORG   0.99
   Amazon               ORG   0.99
   Amazon               LOC   1.00

Jordan met Washington in Washington.
   Jordan               PER   0.92
   Washington           LOC   0.98
   Washington           LOC   1.00

OpenAI and Anthropic released new models in 2026.
   OpenAI               ORG   0.99
   Anthropic            ORG   0.89

Ahmed Hassan works at Vodafone in Alexandria.
   Ahmed Hassan         PER   1.00
   Vodafone             ORG   1.00
   Alexandria           LOC   1.00

barack obama visited cairo.
   (no entities found)

BREAKING: NASA LAUNCHES NEW SATELLITE FROM FLORIDA
   NASA    

In [12]:
tests = [
    "OpenAI and Anthropic released new models in 2026.",
    "Mohamed Salah scored twice for Liverpool at Anfield.",
    "Apple released a new phone while Amazon opened a warehouse near the Amazon river.",
]
for s in tests:
    print(s)
    for e in ner(s):
        print(f"   {e['word']:20} {e['entity_group']:5} {e['score']:.2f}")
    print()

OpenAI and Anthropic released new models in 2026.
   OpenAI               ORG   0.99
   Anthropic            ORG   0.89

Mohamed Salah scored twice for Liverpool at Anfield.
   Mohamed Salah        PER   1.00
   Liverpool            ORG   0.99
   Anfield              LOC   0.98

Apple released a new phone while Amazon opened a warehouse near the Amazon river.
   Apple                ORG   0.99
   Amazon               ORG   0.99
   Amazon               LOC   1.00

